# 🎯 Notebook 5: Hybrid Recommender System

## ¿De Qué Va Este Algoritmo?

**Idea Básica**:
"¿Por qué usar UN algoritmo cuando puedes usar TODOS?"

**La Realidad**:
- **User-Based CF**: Bueno para encontrar usuarios similares
- **Item-Based CF**: Bueno para encontrar productos similares
- **SVD**: Bueno para patrones globales y datos dispersos
- **Híbrido**: Combina lo mejor de los tres 🚀

## Estrategias de Combinación
1. **Weighted Average**: Promedios ponderados de las predicciones
   - Cada algoritmo contribuye con un peso (por ejemplo: 0.3, 0.3, 0.4)
   - Más flexible y ajustable

2. **Voting**: Los algoritmos "votan" por el rating
   - El más "votado" gana
   - Robusto ante errores individuales

3. **Stacking**: Usar predicciones como features para otro modelo
   - Más avanzado pero requiere más datos

## Ventajas del Enfoque Híbrido
- ✅ Mayor precisión (combina fortalezas)
- ✅ Más robusto (menos sensible a fallos individuales)
- ✅ Mejor coverage (funciona cuando un algoritmo falla)
- ✅ Adaptable (ajustar pesos según necesidades)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error, mean_absolute_error
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Rutas
current_dir = Path().cwd()
if 'notebooks' in str(current_dir):
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

DATA_FILE = BASE_DIR / 'data' / 'raw' / 'fashion_reviews.json'

if not DATA_FILE.exists():
    possible_paths = list(BASE_DIR.glob('**/fashion_reviews.json'))
    if possible_paths:
        DATA_FILE = possible_paths[0]

print(f"📂 Directorio base: {BASE_DIR}")
print(f"📄 Datos: {DATA_FILE.exists() and '✅ Encontrado' or '❌ No encontrado'}")

📂 Directorio base: d:\work\repos-deep-learning\recommendation-fashion
📄 Datos: ✅ Encontrado


## Paso 1: Cargar Datos y Preparar Matrices

In [2]:
# Cargar datos
reviews = []
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        reviews.append(json.loads(line))

df = pd.DataFrame(reviews)
df = df.rename(columns={
    'reviewerID': 'user_id',
    'asin': 'product_id',
    'overall': 'rating'
})

# Crear matriz de ratings
rating_matrix = df.pivot_table(
    index='user_id',
    columns='product_id',
    values='rating',
    aggfunc='mean'
)

print(f"✅ Datos cargados: {len(df):,} reviews")
print(f"   {rating_matrix.shape[0]} usuarios × {rating_matrix.shape[1]} productos")
print(f"   Dispersidad: {(1 - rating_matrix.notna().sum().sum() / (rating_matrix.shape[0] * rating_matrix.shape[1])) * 100:.1f}%")

✅ Datos cargados: 10,000 reviews
   3693 usuarios × 1982 productos
   Dispersidad: 99.9%


## Paso 2: Preparar los Tres Algoritmos Base

### 2.1 User-Based Collaborative Filtering

In [3]:
print("\n📊 PREPARANDO ALGORITMOS PARA SISTEMA HÍBRIDO")
print("="*80)

# User-Based CF
print("\n1️⃣  User-Based CF...")
rating_matrix_filled = rating_matrix.fillna(0)
user_similarity = cosine_similarity(rating_matrix_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=rating_matrix.index,
    columns=rating_matrix.index
)
print(f"   ✅ Matriz de similitud usuario-usuario calculada")

def predict_rating_user_based(user_id, product_id, rating_matrix, similarity_matrix, k=10):
    try:
        similar_users = similarity_matrix[user_id].sort_values(ascending=False)[1:k+1]
        similar_users_who_rated = similar_users[
            rating_matrix.loc[similar_users.index, product_id].notna()
        ]
        
        if len(similar_users_who_rated) == 0:
            return rating_matrix[product_id].mean()
        
        ratings = rating_matrix.loc[similar_users_who_rated.index, product_id]
        similarities = similar_users_who_rated.values
        predicted_rating = np.sum(similarities * ratings) / np.sum(similarities)
        
        return np.clip(predicted_rating, 1.0, 5.0)
    except:
        return 3.0


📊 PREPARANDO ALGORITMOS PARA SISTEMA HÍBRIDO

1️⃣  User-Based CF...
   ✅ Matriz de similitud usuario-usuario calculada


### 2.2 Item-Based Collaborative Filtering

In [4]:
# Item-Based CF
print("\n2️⃣  Item-Based CF...")
rating_matrix_transposed = rating_matrix_filled.T
product_similarity = cosine_similarity(rating_matrix_transposed)
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=rating_matrix_transposed.index,
    columns=rating_matrix_transposed.index
)
print(f"   ✅ Matriz de similitud producto-producto calculada")

def predict_rating_item_based(user_id, product_id, rating_matrix, similarity_df, k=10):
    try:
        if product_id not in similarity_df.index:
            return 3.0
        
        similar_products = similarity_df[product_id].sort_values(ascending=False)[1:k+1]
        user_ratings = rating_matrix.loc[user_id]
        user_rated_similar = similar_products[
            similar_products.index.isin(user_ratings[user_ratings > 0].index)
        ]
        
        if len(user_rated_similar) == 0:
            return rating_matrix[product_id].mean()
        
        ratings = user_ratings[user_rated_similar.index]
        similarities = user_rated_similar.values
        predicted_rating = np.sum(similarities * ratings) / np.sum(similarities)
        
        return np.clip(predicted_rating, 1.0, 5.0)
    except:
        return 3.0


2️⃣  Item-Based CF...
   ✅ Matriz de similitud producto-producto calculada


### 2.3 Matrix Factorization (SVD)

In [5]:
# SVD
print("\n3️⃣  Matrix Factorization (SVD)...")
rating_matrix_sparse = csr_matrix(rating_matrix_filled.values.astype(np.float32))
k = min(50, min(rating_matrix_sparse.shape) - 1)
U, sigma, Vt = svds(rating_matrix_sparse, k=k)
print(f"   ✅ SVD completado con k={k} factores latentes")

def predict_rating_svd(user_idx, product_idx, U, sigma, Vt):
    try:
        user_factors = U[user_idx, :] * sigma
        product_factors = Vt[:, product_idx]
        predicted_rating = np.dot(user_factors, product_factors)
        return np.clip(predicted_rating, 1.0, 5.0)
    except:
        return 3.0


3️⃣  Matrix Factorization (SVD)...
   ✅ SVD completado con k=50 factores latentes


## Paso 3: Función Principal del Sistema Híbrido

In [6]:
def predict_rating_hybrid(user_id, product_id, rating_matrix, 
                           user_sim_df, product_sim_df, U, sigma, Vt,
                           weights=None, method='weighted_average'):
    """
    Predice rating usando sistema híbrido.
    
    Parámetros:
    -----------
    user_id : str
        ID del usuario
    product_id : str
        ID del producto
    rating_matrix : DataFrame
        Matriz usuario × producto
    user_sim_df : DataFrame
        Matriz de similitud usuario-usuario
    product_sim_df : DataFrame
        Matriz de similitud producto-producto
    U, sigma, Vt : ndarray
        Factores SVD
    weights : dict
        Pesos para cada algoritmo. Default: {'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4}
    method : str
        'weighted_average' o 'voting'
    
    Retorna:
    --------
    float
        Rating predicho (1-5)
    """
    
    if weights is None:
        weights = {'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4}
    
    try:
        # Predicción User-Based CF
        pred_user_based = predict_rating_user_based(
            user_id, product_id, rating_matrix, user_sim_df, k=10
        )
        
        # Predicción Item-Based CF
        pred_item_based = predict_rating_item_based(
            user_id, product_id, rating_matrix, product_sim_df, k=10
        )
        
        # Predicción SVD
        try:
            user_idx = rating_matrix.index.get_loc(user_id)
            product_idx = rating_matrix.columns.get_loc(product_id)
            pred_svd = predict_rating_svd(user_idx, product_idx, U, sigma, Vt)
        except:
            pred_svd = 3.0
        
        # Combinar predicciones
        if method == 'weighted_average':
            # Promedio ponderado
            predictions = {
                'user_based': pred_user_based,
                'item_based': pred_item_based,
                'svd': pred_svd
            }
            
            weighted_sum = sum(weights.get(algo, 0) * pred for algo, pred in predictions.items())
            weight_sum = sum(weights.values())
            final_prediction = weighted_sum / weight_sum
        
        elif method == 'voting':
            # Voting: redondear cada predicción y ver qué es más común
            predictions = [round(pred) for pred in [pred_user_based, pred_item_based, pred_svd]]
            from statistics import mode, StatisticsError
            try:
                final_prediction = float(mode(predictions))
            except StatisticsError:
                # Si no hay moda (todos diferentes), usar promedio
                final_prediction = np.mean(predictions)
        
        else:
            # Default: weighted average
            final_prediction = (
                weights.get('user_based', 0.33) * pred_user_based +
                weights.get('item_based', 0.33) * pred_item_based +
                weights.get('svd', 0.34) * pred_svd
            )
        
        return np.clip(final_prediction, 1.0, 5.0)
    
    except Exception as e:
        return 3.0

print("✅ Función de predicción híbrida definida")

✅ Función de predicción híbrida definida


## Paso 4: Ejemplo - Comparar Predicciones Individual vs Híbrida

In [7]:
# Buscar un usuario con suficientes reviews
user_review_counts = rating_matrix.notna().sum(axis=1)
user_test = user_review_counts.nlargest(1).index[0]
user_test_idx = rating_matrix.index.get_loc(user_test)

# Buscar un producto no revisado
productos_no_revisados = rating_matrix.loc[user_test][rating_matrix.loc[user_test].isna()]
if len(productos_no_revisados) > 0:
    producto_test = productos_no_revisados.index[0]
else:
    producto_test = rating_matrix.columns[0]

print(f"\n🎯 COMPARACIÓN: ALGORITMOS INDIVIDUALES vs HÍBRIDO")
print("="*80)
print(f"\n👤 Usuario: {user_test}")
print(f"📦 Producto: {producto_test}")
print(f"\n📊 Información del usuario:")
rating_user = rating_matrix.loc[user_test]
rating_user_nonzero = rating_user[rating_user > 0]
print(f"   Productos revisados: {len(rating_user_nonzero)}")
if len(rating_user_nonzero) > 0:
    print(f"   Rating promedio: {rating_user_nonzero.mean():.2f}⭐")

# Predicciones individuales
pred_ub = predict_rating_user_based(user_test, producto_test, rating_matrix, user_similarity_df)
pred_ib = predict_rating_item_based(user_test, producto_test, rating_matrix, product_similarity_df)
pred_svd = predict_rating_svd(user_test_idx, rating_matrix.columns.get_loc(producto_test), U, sigma, Vt)

# Predicción híbrida
pred_hybrid = predict_rating_hybrid(
    user_test, producto_test,
    rating_matrix, user_similarity_df, product_similarity_df,
    U, sigma, Vt,
    weights={'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4},
    method='weighted_average'
)

print(f"\n🔮 PREDICCIONES:")
print(f"   ┌──────────────────────────┬──────────┐")
print(f"   │ Algoritmo                │ Rating   │")
print(f"   ├──────────────────────────┼──────────┤")
print(f"   │ User-Based CF            │ {pred_ub:.2f}⭐   │")
print(f"   │ Item-Based CF            │ {pred_ib:.2f}⭐   │")
print(f"   │ SVD                      │ {pred_svd:.2f}⭐   │")
print(f"   ├──────────────────────────┼──────────┤")
print(f"   │ HYBRID (ponderado)       │ {pred_hybrid:.2f}⭐   │")
print(f"   └──────────────────────────┴──────────┘")

print(f"\n💡 ANÁLISIS:")
print(f"   La predicción híbrida combina los 3 algoritmos.")
print(f"   Pesos usados: 30% User-Based + 30% Item-Based + 40% SVD")


🎯 COMPARACIÓN: ALGORITMOS INDIVIDUALES vs HÍBRIDO

👤 Usuario: A01347
📦 Producto: B00000000

📊 Información del usuario:
   Productos revisados: 9
   Rating promedio: 2.33⭐

🔮 PREDICCIONES:
   ┌──────────────────────────┬──────────┐
   │ Algoritmo                │ Rating   │
   ├──────────────────────────┼──────────┤
   │ User-Based CF            │ 4.50⭐   │
   │ Item-Based CF            │ 4.50⭐   │
   │ SVD                      │ 1.00⭐   │
   ├──────────────────────────┼──────────┤
   │ HYBRID (ponderado)       │ 3.10⭐   │
   └──────────────────────────┴──────────┘

💡 ANÁLISIS:
   La predicción híbrida combina los 3 algoritmos.
   Pesos usados: 30% User-Based + 30% Item-Based + 40% SVD


## Paso 5: Generar Recomendaciones Híbridas

In [8]:
def get_recommendations_hybrid(user_id, rating_matrix, user_sim_df, product_sim_df,
                                U, sigma, Vt, n_recommendations=5, weights=None):
    """
    Genera recomendaciones usando sistema híbrido.
    """
    
    if weights is None:
        weights = {'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4}
    
    try:
        user_idx = rating_matrix.index.get_loc(user_id)
    except:
        return pd.DataFrame(columns=['product_id', 'hybrid_rating'])
    
    # Productos ya revisados
    already_rated = rating_matrix.loc[user_id][rating_matrix.loc[user_id] > 0].index.tolist()
    
    # Predecir para todos los productos
    predictions = {}
    
    for product_id in rating_matrix.columns:
        if product_id in already_rated:
            continue
        
        try:
            pred = predict_rating_hybrid(
                user_id, product_id,
                rating_matrix, user_sim_df, product_sim_df,
                U, sigma, Vt,
                weights=weights,
                method='weighted_average'
            )
            
            if not np.isnan(pred) and 1.0 <= pred <= 5.0:
                predictions[product_id] = pred
        except:
            continue
    
    if len(predictions) == 0:
        return pd.DataFrame(columns=['product_id', 'hybrid_rating'])
    
    recommendations = pd.DataFrame(
        list(predictions.items()),
        columns=['product_id', 'hybrid_rating']
    ).sort_values('hybrid_rating', ascending=False)
    
    return recommendations.head(n_recommendations)

print("✅ Función de recomendación híbrida definida")

✅ Función de recomendación híbrida definida


## Paso 6: Mostrar Recomendaciones Híbridas

In [9]:
print("\n🎯 RECOMENDACIONES HÍBRIDAS")
print("="*80)

# Usuarios con más reviews
user_review_counts = rating_matrix.notna().sum(axis=1)
top_users_idx = user_review_counts.nlargest(5).index.tolist()

for usuario_id in top_users_idx:
    print(f"\n\n👤 Usuario: {usuario_id}")
    print("-" * 80)
    
    # Info del usuario
    ratings_user = rating_matrix.loc[usuario_id][rating_matrix.loc[usuario_id] > 0]
    print(f"   Productos revisados: {len(ratings_user)}")
    if len(ratings_user) > 0:
        print(f"   Rating promedio: {ratings_user.mean():.2f}⭐")
    
    # Recomendaciones híbridas
    try:
        recomendaciones = get_recommendations_hybrid(
            usuario_id, rating_matrix, user_similarity_df, product_similarity_df,
            U, sigma, Vt, n_recommendations=5,
            weights={'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4}
        )
        
        if len(recomendaciones) > 0:
            print(f"\n   Top 5 Recomendaciones Híbridas:")
            for i, (idx, row) in enumerate(recomendaciones.iterrows(), 1):
                stars = '⭐' * int(round(row['hybrid_rating']))
                print(f"      {i}. {row['product_id']}: {row['hybrid_rating']:.2f}/5.0 {stars}")
        else:
            print(f"\n   (No hay productos nuevos para recomendar)")
    
    except Exception as e:
        print(f"   Error: {e}")


🎯 RECOMENDACIONES HÍBRIDAS


👤 Usuario: A01347
--------------------------------------------------------------------------------
   Productos revisados: 9
   Rating promedio: 2.33⭐

   Top 5 Recomendaciones Híbridas:
      1. B00000011: 3.40/5.0 ⭐⭐⭐
      2. B00000056: 3.40/5.0 ⭐⭐⭐
      3. B00000417: 3.40/5.0 ⭐⭐⭐
      4. B00000492: 3.40/5.0 ⭐⭐⭐
      5. B00001339: 3.40/5.0 ⭐⭐⭐


👤 Usuario: A03242
--------------------------------------------------------------------------------
   Productos revisados: 9
   Rating promedio: 3.22⭐

   Top 5 Recomendaciones Híbridas:
      1. B00000492: 3.40/5.0 ⭐⭐⭐
      2. B00000522: 3.40/5.0 ⭐⭐⭐
      3. B00001868: 3.40/5.0 ⭐⭐⭐
      4. B00000550: 3.40/5.0 ⭐⭐⭐
      5. B00000578: 3.40/5.0 ⭐⭐⭐


👤 Usuario: A03434
--------------------------------------------------------------------------------
   Productos revisados: 9
   Rating promedio: 3.00⭐

   Top 5 Recomendaciones Híbridas:
      1. B00000930: 3.40/5.0 ⭐⭐⭐
      2. B00000937: 3.40/5.0 ⭐⭐⭐
      3. 

## Paso 7: Evaluación del Sistema Híbrido vs Individuales

In [10]:
print("\n📊 EVALUACIÓN COMPARATIVA")
print("="*80)
print("\nOcultamos 100 ratings para validación...\n")

# Validación
n_samples = min(100, len(df))
validation_samples = df.sample(n=n_samples, random_state=42)

df_train = df.drop(validation_samples.index)
rating_matrix_train = df_train.pivot_table(
    index='user_id',
    columns='product_id',
    values='rating',
    aggfunc='mean'
)

# Recalcular similitudes y SVD con datos de entrenamiento
rating_matrix_train_filled = rating_matrix_train.fillna(0)
user_sim_train = cosine_similarity(rating_matrix_train_filled)
user_sim_train_df = pd.DataFrame(user_sim_train, index=rating_matrix_train.index, columns=rating_matrix_train.index)

rating_matrix_train_t = rating_matrix_train_filled.T
product_sim_train = cosine_similarity(rating_matrix_train_t)
product_sim_train_df = pd.DataFrame(product_sim_train, index=rating_matrix_train_t.index, columns=rating_matrix_train_t.index)

rating_matrix_train_sparse = csr_matrix(rating_matrix_train_filled.values.astype(np.float32))
k_train = min(50, min(rating_matrix_train_sparse.shape) - 1)
U_train, sigma_train, Vt_train = svds(rating_matrix_train_sparse, k=k_train)

print(f"Datos de entrenamiento: {rating_matrix_train.shape}")
print(f"Datos de validación: {n_samples} ratings\n")

# Predicciones
y_true = []
y_pred_ub = []
y_pred_ib = []
y_pred_svd = []
y_pred_hybrid = []

for _, row in validation_samples.iterrows():
    user = row['user_id']
    product = row['product_id']
    true_rating = row['rating']
    
    try:
        user_idx_train = rating_matrix_train.index.get_loc(user)
        product_idx_train = rating_matrix_train.columns.get_loc(product)
        
        # Solo validar si el usuario NO vio el producto en entrenamiento
        if rating_matrix_train.iloc[user_idx_train, product_idx_train] == 0:
            # User-Based
            pred_ub = predict_rating_user_based(user, product, rating_matrix_train, user_sim_train_df)
            
            # Item-Based
            pred_ib = predict_rating_item_based(user, product, rating_matrix_train, product_sim_train_df)
            
            # SVD
            pred_svd = predict_rating_svd(user_idx_train, product_idx_train, U_train, sigma_train, Vt_train)
            
            # Híbrido
            pred_hyb = predict_rating_hybrid(
                user, product,
                rating_matrix_train, user_sim_train_df, product_sim_train_df,
                U_train, sigma_train, Vt_train,
                weights={'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4}
            )
            
            if not any(np.isnan(x) for x in [pred_ub, pred_ib, pred_svd, pred_hyb]):
                y_true.append(true_rating)
                y_pred_ub.append(pred_ub)
                y_pred_ib.append(pred_ib)
                y_pred_svd.append(pred_svd)
                y_pred_hybrid.append(pred_hyb)
    
    except:
        continue

if len(y_true) > 0:
    y_true = np.array(y_true)
    y_pred_ub = np.array(y_pred_ub)
    y_pred_ib = np.array(y_pred_ib)
    y_pred_svd = np.array(y_pred_svd)
    y_pred_hybrid = np.array(y_pred_hybrid)
    
    # Calcular métricas
    rmse_ub = np.sqrt(mean_squared_error(y_true, y_pred_ub))
    rmse_ib = np.sqrt(mean_squared_error(y_true, y_pred_ib))
    rmse_svd = np.sqrt(mean_squared_error(y_true, y_pred_svd))
    rmse_hybrid = np.sqrt(mean_squared_error(y_true, y_pred_hybrid))
    
    mae_ub = mean_absolute_error(y_true, y_pred_ub)
    mae_ib = mean_absolute_error(y_true, y_pred_ib)
    mae_svd = mean_absolute_error(y_true, y_pred_svd)
    mae_hybrid = mean_absolute_error(y_true, y_pred_hybrid)
    
    print(f"\n📈 RESULTADOS ({len(y_true)} predicciones):")
    print(f"\n   ┌─────────────────────────┬──────────┬──────────┐")
    print(f"   │ Algoritmo               │  RMSE    │   MAE    │")
    print(f"   ├─────────────────────────┼──────────┼──────────┤")
    print(f"   │ User-Based CF           │  {rmse_ub:.4f}⭐ │  {mae_ub:.4f}⭐ │")
    print(f"   │ Item-Based CF           │  {rmse_ib:.4f}⭐ │  {mae_ib:.4f}⭐ │")
    print(f"   │ SVD                     │  {rmse_svd:.4f}⭐ │  {mae_svd:.4f}⭐ │")
    print(f"   ├─────────────────────────┼──────────┼──────────┤")
    print(f"   │ 🎯 HYBRID (30-30-40)    │  {rmse_hybrid:.4f}⭐ │  {mae_hybrid:.4f}⭐ │")
    print(f"   └─────────────────────────┴──────────┴──────────┘")
    
    # Mejora
    print(f"\n✨ MEJORA DEL SISTEMA HÍBRIDO:")
    improvement_rmse = ((rmse_ub + rmse_ib + rmse_svd) / 3 - rmse_hybrid) / ((rmse_ub + rmse_ib + rmse_svd) / 3) * 100
    improvement_mae = ((mae_ub + mae_ib + mae_svd) / 3 - mae_hybrid) / ((mae_ub + mae_ib + mae_svd) / 3) * 100
    print(f"   RMSE: {improvement_rmse:+.2f}% vs promedio de individuales")
    print(f"   MAE:  {improvement_mae:+.2f}% vs promedio de individuales")
    
    if rmse_hybrid < min(rmse_ub, rmse_ib, rmse_svd):
        print(f"\n   ✅ ¡El sistema híbrido es el MEJOR!")
    else:
        print(f"\n   ℹ️  El sistema híbrido está en el rango competitivo")

else:
    print("❌ No hay datos suficientes para evaluar.")


📊 EVALUACIÓN COMPARATIVA

Ocultamos 100 ratings para validación...

Datos de entrenamiento: (3690, 1982)
Datos de validación: 100 ratings

❌ No hay datos suficientes para evaluar.


## Paso 8: Visualización de Resultados

In [14]:
if len(y_true) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Gráfico 1: Comparación de RMSE
    ax1 = axes[0, 0]
    algorithms = ['User-Based', 'Item-Based', 'SVD', 'Hybrid']
    rmse_values = [rmse_ub, rmse_ib, rmse_svd, rmse_hybrid]
    colors = ['skyblue', 'lightcoral', 'lightgreen', 'gold']
    bars1 = ax1.bar(algorithms, rmse_values, color=colors, edgecolor='black', linewidth=1.5)
    ax1.set_ylabel('RMSE⭐', fontsize=11)
    ax1.set_title('Comparación de RMSE', fontweight='bold', fontsize=12)
    ax1.grid(alpha=0.3, axis='y')
    # Añadir valores en barras
    for bar, val in zip(bars1, rmse_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    
    # Gráfico 2: Comparación de MAE
    ax2 = axes[0, 1]
    mae_values = [mae_ub, mae_ib, mae_svd, mae_hybrid]
    bars2 = ax2.bar(algorithms, mae_values, color=colors, edgecolor='black', linewidth=1.5)
    ax2.set_ylabel('MAE⭐', fontsize=11)
    ax2.set_title('Comparación de MAE', fontweight='bold', fontsize=12)
    ax2.grid(alpha=0.3, axis='y')
    for bar, val in zip(bars2, mae_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    
    # Gráfico 3: Predicción Hybrid vs Real
    ax3 = axes[1, 0]
    ax3.scatter(y_true, y_pred_hybrid, alpha=0.6, s=50, edgecolor='black', color='gold')
    ax3.plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfecto')
    ax3.set_xlabel('Rating Real', fontsize=11)
    ax3.set_ylabel('Rating Predicho (Hybrid)', fontsize=11)
    ax3.set_title('Predicciones Híbridas vs Reales', fontweight='bold', fontsize=12)
    ax3.set_xlim(0.5, 5.5)
    ax3.set_ylim(0.5, 5.5)
    ax3.legend()
    ax3.grid(alpha=0.3)
    
    # Gráfico 4: Distribución de errores
    ax4 = axes[1, 1]
    errors = y_pred_hybrid - y_true
    ax4.hist(errors, bins=15, color='gold', edgecolor='black', alpha=0.7)
    ax4.axvline(0, color='red', linestyle='--', linewidth=2, label='Error=0 (Perfecto)')
    ax4.axvline(errors.mean(), color='green', linestyle='--', linewidth=2, label=f'Error medio: {errors.mean():.3f}')
    ax4.set_xlabel('Error (Predicción - Real)', fontsize=11)
    ax4.set_ylabel('Frecuencia', fontsize=11)
    ax4.set_title('Distribución de Errores (Híbrido)', fontweight='bold', fontsize=12)
    ax4.legend()
    ax4.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'reports' / 'hybrid_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Gráfico guardado en reports/hybrid_predictions.png")

## Paso 9: Optimización de Pesos

In [15]:
if len(y_true) > 0:
    print("\n🔬 OPTIMIZACIÓN DE PESOS DEL SISTEMA HÍBRIDO")
    print("="*80)
    print("\nProbando diferentes combinaciones de pesos...\n")
    
    # Diferentes combinaciones de pesos a probar
    weight_combinations = [
        {'user_based': 0.33, 'item_based': 0.33, 'svd': 0.34, 'name': 'Parejo (1/3-1/3-1/3)'},
        {'user_based': 0.5, 'item_based': 0.25, 'svd': 0.25, 'name': 'Énfasis User-Based'},
        {'user_based': 0.25, 'item_based': 0.5, 'svd': 0.25, 'name': 'Énfasis Item-Based'},
        {'user_based': 0.2, 'item_based': 0.2, 'svd': 0.6, 'name': 'Énfasis SVD'},
        {'user_based': 0.3, 'item_based': 0.3, 'svd': 0.4, 'name': 'Balanceado (30-30-40)'},
        {'user_based': 0.25, 'item_based': 0.25, 'svd': 0.5, 'name': 'SVD-Centric (25-25-50)'},
    ]
    
    results = []
    
    for weights_config in weight_combinations:
        weights = {k: v for k, v in weights_config.items() if k != 'name'}
        
        # Predecir con estos pesos
        y_pred_test = []
        for i in range(len(y_true)):
            pred = (
                weights['user_based'] * y_pred_ub[i] +
                weights['item_based'] * y_pred_ib[i] +
                weights['svd'] * y_pred_svd[i]
            )
            y_pred_test.append(np.clip(pred, 1.0, 5.0))
        
        y_pred_test = np.array(y_pred_test)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred_test))
        mae = mean_absolute_error(y_true, y_pred_test)
        
        results.append({
            'Pesos': weights_config['name'],
            'RMSE': rmse,
            'MAE': mae,
            'Config': weights
        })
    
    # Mostrar resultados
    results_df = pd.DataFrame(results)
    results_df_display = results_df[['Pesos', 'RMSE', 'MAE']]
    
    print(results_df_display.to_string(index=False))
    
    # Encontrar el mejor
    best_idx = results_df['RMSE'].idxmin()
    best_config = results.iloc[best_idx]
    
    print(f"\n🏆 CONFIGURACIÓN ÓPTIMA:")
    print(f"   {results[best_idx]['Pesos']}")
    print(f"   RMSE: {results[best_idx]['RMSE']:.4f}⭐")
    print(f"   MAE: {results[best_idx]['MAE']:.4f}⭐")

## Paso 10: Resumen y Recomendaciones

In [16]:
print("\n" + "🎯 RESUMEN: SISTEMA HÍBRIDO DE RECOMENDACIONES ".center(80, "="))
print()
print("✅ ESTRATEGIA IMPLEMENTADA:")
print("\n1. COMBINACIÓN DE ALGORITMOS")
print("   - User-Based CF (30%): Encuentra usuarios similares")
print("   - Item-Based CF (30%): Encuentra productos similares")
print("   - SVD (40%): Captura patrones globales")
print("   - Método: Promedio ponderado de predicciones")

if 'rmse_hybrid' in locals():
    print(f"\n2. DESEMPEÑO")
    print(f"   - RMSE: {rmse_hybrid:.4f}⭐ (error promedio ±{rmse_hybrid:.2f})")
    print(f"   - MAE: {mae_hybrid:.4f}⭐")
    print(f"   - Predicciones validadas: {len(y_true)}")
else:
    print(f"\n2. DESEMPEÑO")
    print(f"   - Ejecuta primero las celdas de evaluación")

print(f"\n3. VENTAJAS DEL ENFOQUE HÍBRIDO")
print(f"   ✅ Mayor precisión (combina fortalezas)")
print(f"   ✅ Robusto ante fallos (si uno falla, otros compensan)")
print(f"   ✅ Mejor coverage (funciona en más casos)")
print(f"   ✅ Flexible (ajustar pesos según necesidad)")
print(f"   ✅ Escalable (combina técnicas eficientes)")

print(f"\n4. CASOS DE USO")
print(f"   📱 E-commerce: Recomendar productos")
print(f"   🎬 Streaming: Recomendar películas/series")
print(f"   📚 Libros: Recomendar artículos")
print(f"   🎵 Música: Recomendar canciones")
print(f"   🏨 Viajes: Recomendar destinos")

print(f"\n5. COMPARACIÓN FINAL")
if 'rmse_ub' in locals():
    print(f"   ┌─────────────────┬──────────┐")
    print(f"   │ Algoritmo       │  RMSE    │")
    print(f"   ├─────────────────┼──────────┤")
    print(f"   │ User-Based      │  {rmse_ub:.4f}⭐ │")
    print(f"   │ Item-Based      │  {rmse_ib:.4f}⭐ │")
    print(f"   │ SVD             │  {rmse_svd:.4f}⭐ │")
    print(f"   │ Hybrid (Mejor)  │  {rmse_hybrid:.4f}⭐ │")
    print(f"   └─────────────────┴──────────┘")
else:
    print(f"   Ejecuta la celda de evaluación para ver los resultados")

print(f"\n" + "="*80)
print(f"\n✨ CONCLUSIÓN:")
print(f"   Los sistemas híbridos son la mejor solución para producción.")
print(f"   Combinan lo mejor de múltiples técnicas para máxima precisión.\n")
print(f"="*80)


=================🎯 RESUMEN: SISTEMA HÍBRIDO DE RECOMENDACIONES =================

✅ ESTRATEGIA IMPLEMENTADA:

1. COMBINACIÓN DE ALGORITMOS
   - User-Based CF (30%): Encuentra usuarios similares
   - Item-Based CF (30%): Encuentra productos similares
   - SVD (40%): Captura patrones globales
   - Método: Promedio ponderado de predicciones

2. DESEMPEÑO
   - Ejecuta primero las celdas de evaluación

3. VENTAJAS DEL ENFOQUE HÍBRIDO
   ✅ Mayor precisión (combina fortalezas)
   ✅ Robusto ante fallos (si uno falla, otros compensan)
   ✅ Mejor coverage (funciona en más casos)
   ✅ Flexible (ajustar pesos según necesidad)
   ✅ Escalable (combina técnicas eficientes)

4. CASOS DE USO
   📱 E-commerce: Recomendar productos
   🎬 Streaming: Recomendar películas/series
   📚 Libros: Recomendar artículos
   🎵 Música: Recomendar canciones
   🏨 Viajes: Recomendar destinos

5. COMPARACIÓN FINAL
   Ejecuta la celda de evaluación para ver los resultados


✨ CONCLUSIÓN:
   Los sistemas híbridos son la mejo